# Search caps by depth — round 1 only

> **Scope.** This notebook runs the **round-1 fan-out only**, at each depth. No
> continuation rounds, and none of the reformulate / snowball / suggest / diversity
> arms, which only start at round 2 (`search_loop.py`, the `round_index >= 2` branch).
> Round 1 has the identical plan shape at all three depths, so **the only thing
> differing between rapid, standard and deep below is the cap values** — which is what
> this notebook is for. For the multi-round behaviour see
> `search_rounds_and_arms.ipynb`.

Runs the **same query** through `rapid`, `standard` and `deep`, and reports how many
records survive each stage of the acquisition pipeline:

```
fetch          every planned query runs to completion (no clock at any depth)
merge          rank-interleave records across queries, per backend
dedupe         drop unusable (no title) + already-acquired
trim           top N per backend  (record_cap_per_backend: 50/100/200)
save + embed   the survivors become rows
```

Merge changes the **order**, not the count — it decides *which* records the trim keeps.
Every other stage removes records, so the counts below form a funnel.

**This is a live run.** It calls the real OpenAlex and Overton APIs and the real
query-generation model, and it writes rows to whatever `DATABASE_URL` points at.
Screening is **not** run here, so there is no screening spend — LLM cost is three
query-generation calls, a few cents.

In [1]:
import os
import sys
import uuid
from datetime import UTC, datetime
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd()
while not (REPO / "backend").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "backend" / "src"))
load_dotenv(REPO / ".env")

MISSING = [k for k in ("OPENALEX_API_KEY", "OVERTON_API_KEY", "OPENAI_API_KEY")
           if not os.environ.get(k)]
print("repo:", REPO)
print("database:", (os.environ.get("DATABASE_URL") or "UNSET").rsplit("/", 1)[-1])
print("missing keys:", MISSING or "none")

repo: /Users/rosie.oxbury/Documents/git_repos/policy_atlas
database: policy_atlas
missing keys: none


In [2]:
from policy_atlas.core.db import get_engine
from policy_atlas.core.hashing import content_hash
from policy_atlas.core.schema import evidence_scope, project, runs
from policy_atlas.evidence_base.sourcing import search_generation, search_live
from policy_atlas.evidence_base.sourcing.acquire import (
    AcquireContext,
    _MAPPERS,
    _chunk_text,
    _interleave,
    acquire_sources,
)
from policy_atlas.evidence_base.sourcing.search_loop import (
    DEPTH_CONSTANTS,
    ExecutedCall,
    _rapid_plans,
)
from policy_atlas.evidence_base.sourcing.search_prompts import (
    QueriesPayload,
    validated_queries,
)

QUERY = (
    "interventions to reduce consumption of high fat, sugar, and salt (HFSS) foods"
)
DEPTHS = ["rapid", "standard", "deep"]
engine = get_engine()
print(QUERY)

interventions to reduce consumption of high fat, sugar, and salt (HFSS) foods


## The caps at each depth

Two different caps, doing two different jobs. `result_cap_per_backend` bounds **one
call** — how much a single query may fetch. `record_cap_per_backend` bounds **the round**
— how many documents each backend may contribute after the merge and dedupe. Only the
second one bounds what gets paid for downstream.

In [3]:
rows = []
for depth in DEPTHS:
    c = DEPTH_CONSTANTS[depth]
    rows.append((
        depth,
        c["result_cap_per_backend"],
        c["record_cap_per_backend"],
        c["round_cap"],
        c["http_budget"]["openalex"],
        c["http_budget"]["overton"],
    ))

head = ("depth", "per call", "per round", "rounds", "oa calls", "ov calls")
print("{:<10} {:>9} {:>10} {:>7} {:>9} {:>9}".format(*head))
for r in rows:
    print("{:<10} {:>9} {:>10} {:>7} {:>9} {:>9}".format(*r))

depth       per call  per round  wall clock  oa calls  ov calls
rapid             50       none          30        20         5
standard          75        150        none        30         8
deep             100        200        none        50        15


## Stage 1 — fetch

One query-generation call produces up to `N_QUERIES` keyword queries plus Overton
paraphrases. `_rapid_plans` turns those into planned calls: for OpenAlex each query
becomes three (plain, + systematic-review clause, + RCT clause); Overton gets the
verbatim intent plus its paraphrases.

Every planned call runs to completion — no depth has a time budget (task 029
removed the wall clock everywhere; its breach used to silently cost the whole
Overton leg when OpenAlex ran long).

In [4]:
def generate(generation):
    """One query-generation call. Returns (queries, overton_paraphrases)."""
    wire, _usage = generation.generate_queries(QueriesPayload(intent=QUERY))
    return validated_queries(wire)


def fetch(depth, generation, backends, queries=None, paraphrases=None):
    """Run every planned call for one depth. Returns (executed_calls, per_call_rows, queries).

    Pass `queries`/`paraphrases` to reuse ONE generation across depths. Without
    that, each depth searches with a different LLM query set and the depths are
    not comparable — deep can then "return less" than standard purely by query
    luck, despite having strictly larger caps.
    """
    constants = DEPTH_CONSTANTS[depth]
    if queries is None:
        queries, paraphrases = generate(generation)

    executed, per_call = [], []
    for backend in backends:
        budget = constants["http_budget"][backend.name]
        plans = _rapid_plans(
            backend_name=backend.name,
            intent=QUERY,
            queries=queries,
            overton_paraphrases=paraphrases,
        )
        for plan in plans[:budget]:
            try:
                records = backend.search(
                    plan.query, max_results=constants["result_cap_per_backend"]
                )
                status, error = "ok", None
            except Exception as exc:                      # noqa: BLE001
                records, status, error = [], "error", str(exc)
            executed.append(ExecutedCall(
                backend_name=backend.name,
                verb="search",
                query=plan.query,
                query_origin=plan.query_origin,
                wire_params={},
                records=records,
                status=status,
                error=error,
            ))
            per_call.append((backend.name, plan.query_origin, plan.query, len(records), status))
    return executed, per_call, queries

In [5]:
import time

generation = search_generation.OpenAISearchGenerationBackend()

# Generate ONCE and share across depths: the only difference between the three
# runs below should be the caps, not the queries.
QUERIES, PARAPHRASES = generate(generation)
print(f"{len(QUERIES)} generated queries, {len(PARAPHRASES)} Overton paraphrases\n")

FETCHED = {}
for depth in DEPTHS:
    started = time.monotonic()
    backends = search_live.live_search_backends()
    executed, per_call, queries = fetch(depth, generation, backends, QUERIES, PARAPHRASES)
    FETCHED[depth] = dict(
        executed=executed, per_call=per_call, queries=queries,
        seconds=time.monotonic() - started,
    )
    returned = sum(len(c.records) for c in executed)
    by_backend = {}
    for call in executed:
        by_backend[call.backend_name] = by_backend.get(call.backend_name, 0) + len(call.records)
    print(f"{depth:9s} {len(executed):3d} calls  {returned:5d} records  "
          f"{FETCHED[depth]['seconds']:6.1f}s   {by_backend}")

2026-08-05 13:46:17 [info     ] search_generation.queries.usage cached_tokens=0 completion_tokens=124 prompt_tokens=634 total_tokens=758
5 generated queries, 2 Overton paraphrases

rapid      18 calls    652 records    22.0s   {'openalex': 502, 'overton': 150}
standard   18 calls    877 records    20.1s   {'openalex': 652, 'overton': 225}
deep       18 calls   1102 records    17.8s   {'openalex': 802, 'overton': 300}


### Per-query detail

If a backend shows **zero calls** here, that is the starvation failure this change was
made to remove — check it never happens.

In [6]:
depth = "deep"
print(f"generated queries ({len(FETCHED[depth]['queries'])}):")
for q in FETCHED[depth]["queries"]:
    print("  -", q[:110])

print(f"\ncalls at depth={depth}:")
print("  {:9s} {:14s} {:>8s}  {}".format("backend", "origin", "records", "query"))
for backend_name, origin, query, n, status in FETCHED[depth]["per_call"]:
    flag = "" if status == "ok" else "  [ERROR]"
    print(f"  {backend_name:9s} {origin:14s} {n:8d}  {query[:60]}{flag}")

generated queries (5):
  - HFSS foods consumption reduction intervention
  - high fat sugar salt food intake intervention
  - junk food reduction policy OR intervention
  - ultra-processed food consumption reduction
  - healthy eating intervention food choice

calls at depth=deep:
  backend   origin          records  query
  openalex  generated            13  HFSS foods consumption reduction intervention
  openalex  variant_sr            0  (HFSS foods consumption reduction intervention) AND ("system
  openalex  variant_rct           0  (HFSS foods consumption reduction intervention) AND ("random
  openalex  generated           100  high fat sugar salt food intake intervention
  openalex  variant_sr           43  (high fat sugar salt food intake intervention) AND ("systema
  openalex  variant_rct          40  (high fat sugar salt food intake intervention) AND ("randomi
  openalex  generated           100  junk food reduction policy OR intervention
  openalex  variant_sr           19  (

## Stages 2–5 — merge, dedupe, trim, save

`acquire_sources` does all four in one call, so its counts are the authoritative funnel:

| count | stage |
|---|---|
| `results_returned` | what the providers returned (fetch) |
| `skipped_unusable` | dropped at dedupe — no title, so not screenable |
| `already_acquired` | dropped at dedupe — same record id, DOI, or text as one already held |
| `dropped_over_cap` | dropped at trim — past `record_cap_per_backend` |
| `acquired` | saved and embedded |

Each depth gets its own project, so `already_acquired` counts only duplicates *within*
that depth's own fan-out — the three runs do not shadow each other.

In [7]:
def seed(conn, depth):
    now = datetime.now(UTC)
    project_id, run_id, scope_id = uuid.uuid4(), uuid.uuid4(), uuid.uuid4()
    conn.execute(project.insert().values(
        project_id=project_id, name=f"hfss-{depth}-{now:%Y%m%d-%H%M%S}",
        status="active", created_at=now, updated_at=now,
    ))
    conn.execute(runs.insert().values(
        run_id=run_id, project_id=project_id, status="running", started_at=now,
    ))
    conn.execute(evidence_scope.insert().values(
        evidence_scope_id=scope_id, project_id=project_id, intent=QUERY,
        context={"search": {"depth": depth}}, created_at=now,
    ))
    return project_id, run_id, scope_id


RESULTS = {}
for depth in DEPTHS:
    with engine.begin() as conn:
        project_id, run_id, scope_id = seed(conn, depth)
        counts = acquire_sources(
            conn,
            project_id=project_id,
            run_id=run_id,
            context=AcquireContext(scope_id=scope_id, intent=QUERY, context={}),
            backends=search_live.live_search_backends(),
            executed_calls=FETCHED[depth]["executed"],
            depth=depth,
            record_cap_per_backend=DEPTH_CONSTANTS[depth]["record_cap_per_backend"],
        )
    RESULTS[depth] = dict(counts=counts, project_id=project_id)
    print(f"{depth:9s} acquired={counts['acquired']:5d}  "
          f"stop={counts['stop_condition']}  project={project_id}")

2026-08-05 14:00:06 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=inraefr-ec7857d2cdb562679224c0fdfd17c19b cap=50 tag_count=72
2026-08-05 14:00:06 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=stateofnebraska-e9fb3ab18bfc3d7afec34cb8dfc292e9 cap=50 tag_count=54
2026-08-05 14:00:06 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=scottishgovernment-0d1644bab8e4a03410ad932d8321f849 cap=50 tag_count=57
2026-08-05 14:00:06 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=govuk-d0a491f6e1021ecca1bf51df2e74c058 cap=50 tag_count=51
2026-08-05 14:00:07 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=ukparliament_select-73f556189fae003b3fa96e68c00bdf48 cap=50 tag_count=55
2026-08-05 14:00:13 [info     ] embed.summary                  already_embedded=0 budget_exceeded=0 embedded=495 failed=0 run_id=a23ec49f-3224-4e25-9334-911cc467e11b skipped_no_unit

### The funnel, per depth

In [8]:
def funnel(depth):
    counts = RESULTS[depth]["counts"]
    cap = DEPTH_CONSTANTS[depth]["record_cap_per_backend"]
    print(f"=== {depth}  (per-round cap: {cap if cap is not None else 'none'}) ===")
    print("  {:11s} {:>10s} {:>10s} {:>10s}".format("stage", "openalex", "overton", "total"))

    def line(label, key):
        oa = counts["by_backend"].get("openalex", {}).get(key, 0)
        ov = counts["by_backend"].get("overton", {}).get(key, 0)
        print(f"  {label:11s} {oa:10d} {ov:10d} {oa + ov:10d}")

    line("fetched", "results_returned")
    line("- unusable", "skipped_unusable")
    line("- duplicate", "already_acquired")
    line("- over cap", "dropped_over_cap")
    line("= saved", "acquired")
    kept = counts["acquired"]
    returned = counts["results_returned"]
    pct = (100 * kept / returned) if returned else 0
    print(f"  saved {kept} of {returned} fetched ({pct:.0f}%)")
    print()


for depth in DEPTHS:
    funnel(depth)

=== rapid  (per-round cap: none) ===
  stage         openalex    overton      total
  fetched            502        150        652
  - unusable           0          0          0
  - duplicate         94         63        157
  - over cap           0          0          0
  = saved            408         87        495
  saved 495 of 652 fetched (76%)

=== standard  (per-round cap: 150) ===
  stage         openalex    overton      total
  fetched            652        225        877
  - unusable           0          0          0
  - duplicate         17         86        103
  - over cap         485          0        485
  = saved            150        139        289
  saved 289 of 877 fetched (33%)

=== deep  (per-round cap: 200) ===
  stage         openalex    overton      total
  fetched            802        300       1102
  - unusable           0          0          0
  - duplicate         28        122        150
  - over cap         574          0        574
  = saved            2

RO: this shows that we should probably get rid of wall clock and use a cap for rapid (searches of all depths take <30s to run, but if you don't have a cap for rapid search, you allow it to keep more records than standard or deep search!)

### Side by side

In [9]:
head = ("depth", "fetched", "unusable", "duplicate", "over cap", "saved", "fetch s")
print("{:<10} {:>9} {:>9} {:>10} {:>9} {:>7} {:>8}".format(*head))
for depth in DEPTHS:
    c = RESULTS[depth]["counts"]
    print("{:<10} {:>9} {:>9} {:>10} {:>9} {:>7} {:>8.1f}".format(
        depth,
        c["results_returned"],
        c["skipped_unusable"],
        c["already_acquired"],
        c["dropped_over_cap"],
        c["acquired"],
        FETCHED[depth]["seconds"],
    ))

print("\ninvariant (fetched == unusable + duplicate + over cap + saved):")
for depth in DEPTHS:
    c = RESULTS[depth]["counts"]
    total = (c["skipped_unusable"] + c["already_acquired"]
             + c["dropped_over_cap"] + c["acquired"])
    print(f"  {depth:9s} {c['results_returned']} == {total}  "
          f"{'ok' if total == c['results_returned'] else 'MISMATCH'}")

depth        fetched  unusable  duplicate  over cap   saved  fetch s
rapid            652         0        157         0     495     22.0
standard         877         0        103       485     289     20.1
deep            1102         0        150       574     378     17.8

invariant (fetched == unusable + duplicate + over cap + saved):
  rapid     652 == 652  ok
  standard  877 == 877  ok
  deep      1102 == 1102  ok


## What the merge actually did

The trim keeps the first N of the merged stream, so *where* the merge puts each query's
records decides which queries survive. Rank-interleaving takes every query's best hit,
then every query's second-best, and so on — so each query contributes roughly `cap / n`
records instead of the first few queries filling the cap and the rest contributing
nothing.

The cell below re-runs the merge on the same fetched records to show the contribution
per query. It is an illustration, not the production path: it dedupes on text hash only,
where `acquire_sources` also matches on backend record id and DOI, so its "kept" numbers
run slightly high. The funnel above is authoritative.

In [10]:
depth = "deep"
cap = DEPTH_CONSTANTS[depth]["record_cap_per_backend"]

for backend_name in ("openalex", "overton"):
    calls = [c for c in FETCHED[depth]["executed"]
             if c.backend_name == backend_name and c.status == "ok"]
    if not calls:
        print(f"{backend_name}: no successful calls")
        continue

    # Map records the way acquire does, tagging each with its source call index.
    # _interleave only indexes the per-call lists and returns their items, so
    # plain (call_index, content_hash) tuples stand in for candidates here.
    mapper = _MAPPERS[backend_name]
    per_call = []
    for index, call in enumerate(calls):
        mapped_records = []
        for record in call.records:
            mapped = mapper(record)
            if mapped is None:
                continue
            mapped_records.append((index, content_hash(_chunk_text(mapped["envelope"]))))
        per_call.append(mapped_records)

    seen, kept_by_call = set(), {}
    for call_index, chash in _interleave(per_call):
        if cap is not None and sum(kept_by_call.values()) >= cap:
            break
        if chash in seen:
            continue
        seen.add(chash)
        kept_by_call[call_index] = kept_by_call.get(call_index, 0) + 1

    print(f"{backend_name}: {len(calls)} queries, cap {cap if cap else 'none'}, "
          f"kept {sum(kept_by_call.values())}")
    for index, call in enumerate(calls):
        print(f"    q{index:<2d} returned {len(call.records):4d}  "
              f"kept {kept_by_call.get(index, 0):4d}   {call.query[:52]}")
    print()

openalex: 15 queries, cap 200, kept 200
    q0  returned   13  kept   12   HFSS foods consumption reduction intervention
    q1  returned    0  kept    0   (HFSS foods consumption reduction intervention) AND 
    q2  returned    0  kept    0   (HFSS foods consumption reduction intervention) AND 
    q3  returned  100  kept   14   high fat sugar salt food intake intervention
    q4  returned   43  kept   17   (high fat sugar salt food intake intervention) AND (
    q5  returned   40  kept   11   (high fat sugar salt food intake intervention) AND (
    q6  returned  100  kept   17   junk food reduction policy OR intervention
    q7  returned   19  kept   18   (junk food reduction policy OR intervention) AND ("s
    q8  returned   11  kept   11   (junk food reduction policy OR intervention) AND ("r
    q9  returned  100  kept   17   ultra-processed food consumption reduction
    q10 returned   38  kept   16   (ultra-processed food consumption reduction) AND ("s
    q11 returned   38  kept

---

# Repeat runs: how stable are these numbers?

A single run tells you very little. Query generation is non-deterministic, and provider
result counts move. This section repeats the fetch **N_RUNS** times and summarises the
spread.

**Design.** Within each run, one query set is generated and shared by all three depths,
so depth differences are caps only. Across runs the queries are regenerated, so the
spread you see includes generation variability — which is usually the dominant term.

**The search cache is disabled for this section.** `search_live` keeps an in-process
cache with a one-hour TTL keyed on URL + params, so repeated identical queries would
otherwise be served from memory: runs 2..N would show near-zero times and identical
counts. `POLICY_ATLAS_SEARCH_CACHE_TTL_S=0` forces real egress every time, and the
original value is restored afterwards.

**Cost.** Roughly `N_RUNS x 3` depths x ~18 calls = ~550 provider calls plus `N_RUNS`
generation calls. Overton enforces a 1.2 s gap between requests, so expect this to take
tens of minutes. Start with `N_RUNS = 3` if you just want to see it work.

In [ ]:
import time

import pandas as pd

N_RUNS = 3

_prior_ttl = os.environ.get("POLICY_ATLAS_SEARCH_CACHE_TTL_S")
os.environ["POLICY_ATLAS_SEARCH_CACHE_TTL_S"] = "0"   # no cached hits, real egress

rows = []
try:
    for run in range(N_RUNS):
        queries, paraphrases = generate(generation)   # regenerated per run
        for depth in DEPTHS:                       # shared within the run
            backends = search_live.live_search_backends()
            started = time.monotonic()
            executed, _per_call, _q = fetch(depth, generation, backends, queries, paraphrases)
            seconds = time.monotonic() - started

            records = {"openalex": 0, "overton": 0}
            calls = {"openalex": 0, "overton": 0}
            for call in executed:
                records[call.backend_name] += len(call.records)
                calls[call.backend_name] += 1

            rows.append({
                "run": run,
                "depth": depth,
                "seconds": seconds,
                "n_queries": len(queries),
                "openalex_calls": calls["openalex"],
                "overton_calls": calls["overton"],
                "openalex_records": records["openalex"],
                "overton_records": records["overton"],
                "total_records": records["openalex"] + records["overton"],
                "failed_calls": sum(1 for c in executed if c.status == "error"),
            })
            print(f"run {run:2d}  {depth:9s} {seconds:6.1f}s  "
                  f"oa={records['openalex']:5d}  ov={records['overton']:4d}"
                  + ("  [errors]" if rows[-1]["failed_calls"] else ""))
finally:
    if _prior_ttl is None:
        os.environ.pop("POLICY_ATLAS_SEARCH_CACHE_TTL_S", None)
    else:
        os.environ["POLICY_ATLAS_SEARCH_CACHE_TTL_S"] = _prior_ttl

RUNS = pd.DataFrame(rows)
print(f"\n{len(RUNS)} observations over {RUNS['run'].nunique()} runs")
RUNS.head()

## Summary across runs

Median and IQR are the ones to read — with 10 runs a single slow call or an unlucky
query set moves the mean a long way.

In [ ]:
def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)


METRICS = ["seconds", "openalex_records", "overton_records", "total_records"]

summary = (
    RUNS.groupby("depth")[METRICS]
    .agg(["median", iqr, "mean", "std"])
    .reindex(DEPTHS)
    .round(1)
)
summary

In [ ]:
for metric in METRICS:
    print(f"--- {metric} ---")
    print("  {:<10} {:>9} {:>9} {:>9} {:>9} {:>9}".format(
        "depth", "median", "IQR", "mean", "std", "min-max"))
    for depth in DEPTHS:
        s = RUNS.loc[RUNS["depth"] == depth, metric]
        print("  {:<10} {:>9.1f} {:>9.1f} {:>9.1f} {:>9.1f} {:>9}".format(
            depth, s.median(), iqr(s), s.mean(), s.std(),
            f"{s.min():.0f}-{s.max():.0f}"))
    print()

## Paired comparison: does deep ever return less than standard?

Because each run shares one query set across depths, the depths can be compared **within
a run**. Deep has strictly larger caps than standard (100 vs 75 per call, 50 vs 30
calls), so `deep - standard` should never be negative here.

If it is negative, the cause is provider-side variation between the two calls — the same
query asked twice minutes apart — not the caps. If it is *zero*, the caps are not
binding: both depths are returning everything the queries matched, and the constraint is
query generation, not the budget.

In [ ]:
paired = RUNS.pivot(index="run", columns="depth", values="openalex_records")[DEPTHS]
paired["deep - standard"] = paired["deep"] - paired["standard"]
paired["standard - rapid"] = paired["standard"] - paired["rapid"]
print(paired.to_string())
print()
negatives = int((paired["deep - standard"] < 0).sum())
zeros = int((paired["deep - standard"] == 0).sum())
print(f"deep < standard in {negatives}/{len(paired)} runs")
print(f"deep == standard in {zeros}/{len(paired)} runs  "
      f"(caps not binding if high)")

## Notes on what you are looking at

- **`dropped_over_cap == 0`** at a depth means the cap never bound: the providers did not
  return enough after dedupe to reach it. The constraint is then upstream, in query
  generation, not in the cap.
- **Every depth has a per-round cap** (task 029, owner-set): rapid 50 / standard 100 /
  deep 200 per backend per round. There is no wall clock at any depth.
- **`already_acquired` grows with fan-out width** — the SR and RCT variants of a query
  return many of the same works. That is the dedupe doing its job, not waste.
- The saved rows are real. Each depth's `project_id` is printed above if you want to
  inspect or delete them.